# CrossSkinNet: তিনটি shared class ও duplicate review

Development: PAD+HAM; external: PH2। Classes: melanoma, basal cell carcinoma, melanocytic nevus। রোগ নির্ণয় বা similar lesion দেখে duplicate অনুমান করবেন না। এই notebook training চালাবে না। PAD, HAM10000 ও PH2 Add Input দিয়ে attach করুন; Internet On লাগবে clone/install-এর জন্য।

In [ ]:
from pathlib import Path
import subprocess, sys, os

repo = Path("/kaggle/working/Codex")
if not repo.exists():
    subprocess.run(["git", "clone", "https://github.com/mahabubhasanmahin/Codex.git", str(repo)], check=True)
elif (repo / ".git").exists():
    subprocess.run(["git", "pull", "--ff-only", "origin", "main"], cwd=repo, check=True)
else:
    raise RuntimeError("Existing Codex directory is not a Git checkout; preserve and inspect it first")
os.chdir(repo)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "Pillow", "PyYAML", "numpy", "ImageHash", "openpyxl", "ipywidgets"], check=True)


## Merge ও selected-label review প্রস্তুত
সব raw PH2 image duplicate reference থাকবে। PH2 atypical nevi evaluation থেকে বাদ। Existing fingerprint-valid hash cache থাকলে reuse করবে।

In [ ]:
subprocess.run([sys.executable, "scripts/01_merge.py", "--config", "configs/default.yaml", "--schemas", "configs/dataset_schemas.yaml"], check=True)
subprocess.run([sys.executable, "scripts/04_prepare_review.py", "--config", "configs/default.yaml", "--mapping", "configs/label_mapping.csv"], check=True)


## নতুন counts দেখুন
এগুলো preview, final training counts নয়।

In [ ]:
import json, yaml, csv
cfg = yaml.safe_load(Path("configs/default.yaml").read_text())
folder = Path(cfg["paths"]["output_root"]) / "review_preparation"
print((folder / "selection_summary.json").read_text())
with (folder / "selected_class_counts.csv").open() as handle:
    for row in csv.DictReader(handle):
        print(row)


## এক জোড়া করে review
Duplicate = একই image-এর copy/transform বলে নিশ্চিত; Different = আলাদা image; Unsure = নিশ্চিত নন। প্রতিটি decision save হবে। Previous/Next শুধু navigation; edited notes save করতে decision button চাপুন। সবগুলোকে একসঙ্গে True/False করবেন না।

In [ ]:
sys.path.insert(0, str(repo / "src"))
from IPython.display import display
from crossskinnet.data.review_ui import launch_review
panel = launch_review(Path("configs/default.yaml"))
display(panel)


## Review-এর পরে relevance refresh
এটি optional refresh cell; review চলার মাঝখানে Run All করে panel rebuild করবেন না। পুরোনো panel বন্ধ করে নতুন panel খুলুন। Site mapping ও actual stages 03–05 এখনো বাকি।

In [ ]:
subprocess.run([sys.executable, "scripts/04_prepare_review.py", "--config", "configs/default.yaml", "--mapping", "configs/label_mapping.csv"], check=True)
panel.close()
panel = launch_review(Path("configs/default.yaml"))
display(panel)


## এরপর
`docs/RUN_ON_KAGGLE.md` অনুযায়ী site inventory/mapping, metadata normalization, actual dedup ও dataset build চালাতে হবে। Preview directory থেকে model train/build করবেন না।